# Lab F3. Combining forecasts

When several models each capture part of the truth, an average of their forecasts often beats every one of
them, and simple averages are hard to beat with estimated weights (the "forecast combination puzzle"). Bayesian
model averaging weights the models by how well the data support them.

**Help:** for any pymetrix function, type its name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call. The last cell lists every pymetrix function, class and method this lab uses.

Topics: equal weights, Bates-Granger (inverse MSE) and Granger-Ramanathan (regression) weights estimated in real
time; the linear pool of densities; Bayesian model averaging with Zellner's g-prior (Elliott and Timmermann,
chapter 14).

The data: 200 quarters of $y_t = 0.8\,f_{t-1} + \varepsilon_t$, where $f$ is an AR(1) factor that is not
observed. Three indicators measure it with error, $x_{j,t} = f_t + u_{j,t}$ ($j = 1, 2, 3$, with noise of
different sizes), and two more, $x_4$ and $x_5$, are unrelated noise.

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd

from pymetrix import BMA, Dataset, Equation, backtest, combine_forecasts

rng = np.random.default_rng(14)
n = 200
f = np.zeros(n)
for t in range(1, n):
    f[t] = 0.7 * f[t - 1] + rng.standard_normal()
noise = rng.standard_normal((n, 5)) * [0.8, 1.0, 1.2, 1.0, 1.0]
x = np.column_stack([f, f, f, np.zeros(n), np.zeros(n)]) + noise
y = np.r_[0.0, 0.8 * f[:-1]] + rng.standard_normal(n)
frame = pd.DataFrame(np.column_stack([y, x]), columns=["y", "x1", "x2", "x3", "x4", "x5"],
                     index=pd.period_range("1975Q1", periods=n, freq="Q"))
data = Dataset.from_frame(frame, source="simulated")

## 1. Three partial models

Each model uses one of the three indicators, so each sees the factor through its own noise. One-step forecasts
from 2000Q1 to 2024Q3, each model re-estimated at every origin.

In [ ]:
origins = ("2000Q1", "2024Q3")
members = {f"x{j}": backtest(Equation(f"y ~ x{j}[-1]").fit(data), data, origins, horizon=1, label=f"x{j}") for j in (1, 2, 3)}
pd.DataFrame({name: bt.scores(["y"]).iloc[0] for name, bt in members.items()}).T[["RMSE", "MAE"]]

## 2. Combinations

The weights are estimated at each origin from the forecasts whose outcomes were known by then, so the combination
is a real-time forecast too. Equal weights need nothing estimated.

In [ ]:
combos = {method: combine_forecasts(members, method=method, label=method) for method in ("equal", "inverse_mse", "regression")}
everything = {**members, **combos}
pd.DataFrame({name: bt.scores(["y"]).iloc[0] for name, bt in everything.items()}).T[["RMSE", "MAE"]]

**Check it yourself.** The equal-weight combination is the average of the three forecasts, origin by origin.

In [ ]:
by_hand = pd.concat([bt.errors.set_index("target")["forecast"] for bt in members.values()], axis=1).mean(axis=1)
combined = combos["equal"].errors.set_index("target")["forecast"]
assert np.allclose(by_hand.loc[combined.index], combined)
print("the equal-weight combination is the simple average")

The weights the regression method used, on average over the origins. The least noisy indicator ($x_1$) should
get the most weight; but the weights are estimated from a few dozen forecasts, and it shows.

In [ ]:
combos["regression"].weights_table()

Each member's forecast has a normal predictive density (from its standard error), so the combination's density
is their mixture, the linear pool: wider than the members when they disagree. Score them with the log score
(higher is better) and the CRPS (lower is better). A mixture needs weights that are not negative, so the
regression combination has a density only at origins where all its weights are positive: it is left out here.

In [ ]:
pd.DataFrame({name: bt.density_scores(["y"]).iloc[0] for name, bt in everything.items() if name != "regression"}).T

**Questions.**

1. Do the estimated weights beat equal weights? Try `min_history=40` for the regression weights. Why do estimated
   weights so often lose to the simple average?
2. Test whether the equal-weight combination is significantly better than the best single model
   (`combos["equal"].compare(members["x1"], "y", 1)`) and whether `x1` encompasses `x2`.

## 3. Bayesian model averaging

Instead of combining fixed models, average over every subset of the five indicators, weighting each model by its
posterior probability. With Zellner's g-prior the marginal likelihood has a closed form, so all 32 models are
computed exactly.

In [ ]:
bma = BMA("y", candidates=("x1[-1]", "x2[-1]", "x3[-1]", "x4[-1]", "x5[-1]")).fit(data)
bma.summary()

**Check it yourself.** With the g-prior and an intercept in every model, the Bayes factor of a model with $k$
regressors and fit $R^2$ against the intercept-only model is
$(1 + g)^{(n - 1 - k)/2} \,/\, [1 + g(1 - R^2)]^{(n - 1)/2}$ (Liang et al. 2008). Fit every subset by least
squares, weight it, and add up the probabilities of the models that include each indicator.

In [ ]:
import statsmodels.api as sm

names = ["x1", "x2", "x3", "x4", "x5"]
lagged = frame[names].shift(1).iloc[1:]
target = frame["y"].iloc[1:]
nobs, g = len(target), bma.g
log_bf, members_of = [], []
for k in range(6):
    for subset in combinations(range(5), k):
        r2 = sm.OLS(target, sm.add_constant(lagged.iloc[:, list(subset)])).fit().rsquared if subset else 0.0
        log_bf.append((nobs - 1 - k) / 2 * np.log1p(g) - (nobs - 1) / 2 * np.log1p(g * (1 - r2)))
        members_of.append(subset)
post = np.exp(np.array(log_bf) - max(log_bf))
post /= post.sum()
inclusion = [sum(p for p, s in zip(post, members_of) if j in s) for j in range(5)]
assert nobs == bma.nobs and np.allclose(inclusion, bma.inclusion)
{name: round(float(p), 3) for name, p in zip(names, inclusion)}

Model averaging is a forecasting method too: backtest it, re-estimating the posterior at every origin, and add it
to the comparison.

In [ ]:
bt_bma = backtest(bma, data, origins, horizon=1, label="BMA")
pd.DataFrame({name: bt.scores(["y"]).iloc[0] for name, bt in {**everything, "BMA": bt_bma}.items()}).T[["RMSE", "MAE"]]

**Questions.**

3. $x_5$ is pure noise, yet its inclusion probability is near one half, while $x_3$, which does measure the
   factor, is low. How can that happen? Change the seed in the first cell and refit: how stable are the
   inclusion probabilities? What does `g=1` do to them?
4. Model averaging puts most weight on $x_1$ and $x_2$; the equal-weight combination averages all three
   indicators. Which forecasts better here, and is the difference significant (`bt_bma.compare(combos["equal"],
   "y", 1)`)? When would you expect averaging over models to beat a simple average, and when not?

In the app: **Forecast → Combine forecasts** picks the backtests and the weighting; **Estimate → Bayesian model
averaging** fits the model average, which **Forecast → Backtest** can then evaluate.

## pymetrix used in this lab

Every pymetrix function, class and method this lab calls, with its arguments and the first line of its documentation. For the full documentation, type the name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call.

### From `pymetrix`

| | |
|---|---|
| `backtest(fitted, data, origins, horizon, vintages, variable, label, window)` | Re-estimate at each origin from `origins[0]` to `origins[1]` and forecast `horizon` periods ahead. |
| `BMA(outcome, candidates, always, g)` | Bayesian model averaging over every subset of candidate regressors, with Zellner's g-prior. |
| `combine_forecasts(backtests, method, min_history, label)` | Combine the forecasts of several backtests into one, and evaluate it as a backtest of its own. |
| `Equation(text, params, family, priors, name, errors, variance)` | An equation written as text: `"c ~ y + c[-1]"` to estimate, or `"y = c + i + g"` (an identity). |

### Methods and properties of the results

| | |
|---|---|
| `Backtest.density_scores(variables)` | By variable and horizon: n, the average log score (higher is better), the average CRPS (lower is better, in the variable's units), and the shares of outcomes inside the central 50% and 90% predictive intervals (0.5 and 0.9 when the densities are well calibrated). |
| `Backtest.scores(variables)` | n, RMSE, MAE and mean error by variable and horizon (a two-level index). |
| `Backtest.weights_table()` | A combination's average weight on each member, by variable and horizon. |
| `BMA.fit(data, span)` | Fit every model on the periods of `span` (first, last) and weight each by its posterior probability. |
| `Dataset.from_frame(df, time, entity, freq, variables, meta, source)` | Build a Dataset from a DataFrame, naming its time and entity columns. |
| `Equation.fit(data, span, where, method, weights, missing)` | Estimate the equation on `data`: by least squares unless `method` says otherwise (`OLS(cov="hac")`, `IV(...)`, `ML()`, `Ridge()`, ...). |
| `FittedBMA.summary()` | Posterior inclusion probabilities and the model-averaged coefficients (means and sds), most probable first. |